# **PERSONALIZED IMAGES RECOMMENDATION SYSTEM USING FAISS :**

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install git+https://github.com/openai/CLIP.git

In [ ]:
!pip install faiss-cpu

## Import libraries :

In [ ]:
import os
import clip
import faiss
import torch
import requests
import gradio as gr
from io import BytesIO
from PIL import Image
from typing import List
from langchain.embeddings.base import Embeddings

## Loading images dataset : 

In [ ]:
def load_images(directory):
    images_list=[]
    images_path=[]
    for root,dirs,files in os.walk(directory):
        for file in files :
            if file.lower().endswith((".jpg",".jpeg",".png")):
                path=os.path.join(root,file)
                try :
                    img=Image.open(path).convert("RGB")
                    images_list.append(img)
                    images_path.append(path)
                except Exception as e :
                    print(f"Error occured {e} while loading {directory} directory")
    print(f"Successfully loaded images from directory {directory}")
    return images_list,images_path

In [ ]:
images_list=[]
images_path=[]

In [ ]:
images_list,images_path=load_images("/kaggle/input/caltech-101/caltech-101")

lists,paths=load_images("/kaggle/input/animals10/raw-img")
images_list.extend(lists)
images_path.extend(paths)

lists,paths=load_images("/kaggle/input/intel-image-classification")
images_list.extend(lists)
images_path.extend(paths)



## Converting into Embeddings Using CLIP Model:

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model,preprocess=clip.load("ViT-B/32",device=device)

In [ ]:
class CLIPEmbeddings:
    def __init__(self,model,preprocess,device):
        self.model=model,
        self.preprocess=preprocess,
        self.device=device
        